# NB03: Statistical Inference

## Welfare Scheme Participation Analysis

OLS regression with HC3 robust standard errors to identify predictors of:
1. **MGNREGA participation gap**
2. **Health insurance coverage gap**

**Key methodological choice**: We estimate both base models (demographics only) and fixed-effects models (demographics + state dummies). Welfare scheme administration is state-level, so state fixed effects absorb the bulk of variance. The variance decomposition reveals whether demographics or state administration drives gaps.

Anomaly detection uses the FE model - districts where gaps exceed **within-state** predictions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.config import MASTER_DATA
from src.feature_engineer import build_features
from src.statistical_analysis import (
    run_full_analysis, save_regression_results, save_anomaly_districts,
    MGNREGA_FEATURES, HEALTH_INS_FEATURES, variance_decomposition
)
from src.visualization import (
    setup_plot_style, save_figure, plot_coefficient_forest,
    plot_residual_distribution, plot_feature_importance_bar,
    plot_gap_comparison
)

setup_plot_style()
master = pd.read_csv(MASTER_DATA)
df = build_features(master)
print(f'Dataset: {df.shape[0]} districts')

## 1. Feature Selection & VIF

In [ ]:
from src.statistical_analysis import compute_vif

print('MGNREGA features:', MGNREGA_FEATURES)
vif_m = compute_vif(df, MGNREGA_FEATURES)
print(vif_m.to_string(index=False))

In [ ]:
print('Health Insurance features:', HEALTH_INS_FEATURES)
vif_h = compute_vif(df, HEALTH_INS_FEATURES)
print(vif_h.to_string(index=False))

## 2. Variance Decomposition: State vs Demographics

In [ ]:
vdec_m = variance_decomposition(df, 'mgnrega_gap_pct', MGNREGA_FEATURES)
vdec_h = variance_decomposition(df, 'health_insurance_gap_pct', HEALTH_INS_FEATURES)

print('=' * 60)
print('VARIANCE DECOMPOSITION')
print('=' * 60)
for name, vdec in [('MGNREGA', vdec_m), ('Health Insurance', vdec_h)]:
    print(f'\n{name}:')
    print(f'  Demographics alone:  R2 = {vdec["r_demographics"]:.3f}')
    print(f'  State FE alone:      R2 = {vdec["r_state_fe"]:.3f}')
    print(f'  Joint (demo+state):  R2 = {vdec["r_joint"]:.3f}')
    print(f'  Demographics add over state: {vdec["demographics_add"]:.3f}')
    print(f'  State adds over demographics: {vdec["state_adds"]:.3f}')

print('\nKEY FINDING: State administration explains the majority of gap variance.')
print('For MGNREGA, demographics add only ~2% over state effects.')
print('For Health Insurance, demographics add ~6% over state effects.')


## 3. MGNREGA Gap: Base Model vs Fixed Effects

In [ ]:
mgnrega_result = run_full_analysis(df, scheme='mgnrega')
print('MGNREGA Base Model (demographics only):')
print(f'  R2 = {mgnrega_result["summary"]["r_squared_base"]:.3f}')
print(f'  adj_R2 = {mgnrega_result["summary"]["adj_r_squared_base"]:.3f}')
print()
print('MGNREGA Fixed Effects Model (demographics + state):')
print(f'  R2 = {mgnrega_result["summary"]["r_squared_fe"]:.3f}')
print(f'  adj_R2 = {mgnrega_result["summary"]["adj_r_squared_fe"]:.3f}')


In [ ]:
print('MGNREGA FE Model: Within-State Demographic Effects')
print(mgnrega_result['model_fe'].summary())

In [ ]:
fig = plot_coefficient_forest(
    mgnrega_result['model_fe'],
    'MGNREGA Gap: Coefficient Forest (FE Model, HC3 robust SE)',
    save_name='mgnrega_fe_coefficient_forest'
)

In [ ]:
fig = plot_feature_importance_bar(
    mgnrega_result['model_fe'], df, MGNREGA_FEATURES,
    'MGNREGA Gap: Standardized Effects (FE Model)',
    save_name='mgnrega_fe_feature_importance'
)

## 4. Health Insurance Gap: Base vs Fixed Effects

In [ ]:
health_result = run_full_analysis(df, scheme='health_insurance')
print('Health Insurance Base Model (demographics only):')
print(f'  R2 = {health_result["summary"]["r_squared_base"]:.3f}')
print(f'  adj_R2 = {health_result["summary"]["adj_r_squared_base"]:.3f}')
print()
print('Health Insurance Fixed Effects Model (demographics + state):')
print(f'  R2 = {health_result["summary"]["r_squared_fe"]:.3f}')
print(f'  adj_R2 = {health_result["summary"]["adj_r_squared_fe"]:.3f}')


In [ ]:
print('Health Insurance FE Model: Within-State Demographic Effects')
print(health_result['model_fe'].summary())

In [ ]:
fig = plot_coefficient_forest(
    health_result['model_fe'],
    'Health Insurance Gap: Coefficient Forest (FE Model, HC3 robust SE)',
    save_name='health_ins_fe_coefficient_forest'
)

In [ ]:
fig = plot_feature_importance_bar(
    health_result['model_fe'], df, HEALTH_INS_FEATURES,
    'Health Insurance Gap: Standardized Effects (FE Model)',
    save_name='health_ins_fe_feature_importance'
)

## 5. Within-State Significant Predictors

In [ ]:
print('=' * 60)
print('SIGNIFICANT WITHIN-STATE PREDICTORS (controlling for state)')
print('=' * 60)

for name, result, features in [('MGNREGA', mgnrega_result, MGNREGA_FEATURES),
                                ('Health Insurance', health_result, HEALTH_INS_FEATURES)]:
    print(f'\n{name}:')
    params = result['model_fe'].params
    pvals = result['model_fe'].pvalues
    sig_count = 0
    for f in features:
        if f in params.index:
            marker = '***' if pvals[f] < 0.01 else '**' if pvals[f] < 0.05 else '*' if pvals[f] < 0.1 else ''
            print(f'  {f}: coef={params[f]:.4f}, p={pvals[f]:.4f} {marker}')
            if pvals[f] < 0.05:
                sig_count += 1
    if sig_count == 0:
        print('  -> No demographic feature is significant within-state')
        print('  -> Gaps are driven by state administration, not demographics')
    else:
        print(f'  -> {sig_count} features significant within-state')


## 6. Residual Distributions (FE Model)

In [ ]:
fig = plot_residual_distribution(
    mgnrega_result['std_residuals'],
    'MGNREGA Gap: Standardized Residuals (FE Model)',
    save_name='mgnrega_fe_residual_dist'
)

In [ ]:
fig = plot_residual_distribution(
    health_result['std_residuals'],
    'Health Insurance Gap: Standardized Residuals (FE Model)',
    save_name='health_ins_fe_residual_dist'
)

## 7. Anomaly Districts (FE Model, Top 5% Residual)

In [ ]:
mgnrega_anomalies = mgnrega_result['anomalies']
health_anomalies = health_result['anomalies']

print(f'MGNREGA anomalies (within-state top 5% residual): {len(mgnrega_anomalies)} districts')
print(f'Health Insurance anomalies (within-state top 5% residual): {len(health_anomalies)} districts')


In [ ]:
print('MGNREGA Top 10 Anomaly Districts (gap above predicted within-state):')
cols = ['state', 'district', 'mgnrega_gap_pct', 'std_residual', 'predicted_gap']
mgnrega_anomalies[cols].head(10).round(2)

In [ ]:
print('Health Insurance Top 10 Anomaly Districts (gap above predicted within-state):')
cols = ['state', 'district', 'health_insurance_gap_pct', 'std_residual', 'predicted_gap']
health_anomalies[cols].head(10).round(2)

In [ ]:
fig = plot_gap_comparison(
    mgnrega_anomalies.head(10), health_anomalies.head(10),
    save_name='anomaly_gap_comparison'
)

## 9. Save Results

## 7. MGNREGA Panel Model (District FE + Year FE, 2011-2021)

In [ ]:
from src.data_loader import load_mgnrega
from src.statistical_analysis import prepare_mgnrega_panel, run_panel_model, PANEL_FEATURES

_, mgnrega_all = load_mgnrega(year='2019-20')
panel = prepare_mgnrega_panel(mgnrega_all)

print(f'Panel: {panel["year"].min()} to {panel["year"].max()}')
print(f'District-years: {len(panel)}')
print(f'Districts: {panel["district"].nunique()}')
print(f'Years: {panel["year"].nunique()}')

In [ ]:
panel_model, panel_summary, panel_features = run_panel_model(panel)

print('MGNREGA Panel Model (District + Year FE + Operational Features + Lag1):')
print(f'  R2 = {panel_summary["r_squared"]:.3f}')
print(f'  adj_R2 = {panel_summary["adj_r_squared"]:.3f}')
print(f'  Obs = {panel_summary["n_obs"]} district-years')
print(f'  Districts = {panel_summary["n_districts"]}')
print(f'  Years = {panel_summary["n_years"]}')


In [ ]:
print('Significant operational predictors (district + year FE):')
for f in panel_features:
    if f in panel_model.params.index and panel_model.pvalues[f] < 0.1:
        sig = '***' if panel_model.pvalues[f] < 0.001 else '**' if panel_model.pvalues[f] < 0.01 else '*' if panel_model.pvalues[f] < 0.05 else '.'
        print(f'  {f}: coef={panel_model.params[f]:.4f}, p={panel_model.pvalues[f]:.6f} {sig}')


## 8. Model Comparison Summary

In [ ]:
print('=' * 70)
print('MODEL COMPARISON: MGNREGA PARTICIPATION GAP')
print('=' * 70)
print()
print(f'1. Demographics only:             R2 = {mgnrega_result["summary"]["r_squared_base"]:.3f}')
print(f'2. Demographics + State FE:       R2 = {mgnrega_result["summary"]["r_squared_fe"]:.3f}')
print(f'3. Expanded + State FE:            R2 = {mgnrega_result["summary"]["r_squared_expanded_fe"]:.3f}')
print(f'4. Panel (District+Year FE+Ops):  R2 = {panel_summary["r_squared"]:.3f}')
print()
print('Progression:')
print('  Step 1->2: State administration adds massive signal')
print('  Step 2->3: MGNREGA operational features (allocation, person-days) add signal')
print('  Step 3->4: Panel + district FE + lags capture within-district dynamics')
print()
print('KEY: Demographics barely matter. Scheme operations + state administration drive gaps.')
print('=' * 70)


In [ ]:
reg_results = save_regression_results(mgnrega_result, health_result)
anomaly_combined = save_anomaly_districts(mgnrega_anomalies, health_anomalies)
print(f'Regression results: {len(reg_results)} rows')
print(f'Anomaly districts: {len(anomaly_combined)} districts')